# Community Detection with LAGO

This notebook shows how to detect temporal communities using `lago_modules`, explore the results, and understand the different parameters.

In [ ]:
from lago import LinkStream, lago_modules

## Create Sample LinkStream

Let's create a sample linkstream with clear community structure.

In [ ]:
def create_sample_linkstream():
    """Create a sample linkstream with clear community structure."""
    ls = LinkStream()
    ls.add_links([
        # Community A: nodes 0, 1, 2 (densely connected at times 0-2)
        (0, 1, 0), (1, 2, 0), (0, 2, 0),
        (0, 1, 1), (1, 2, 1), (0, 2, 1),
        (0, 1, 2), (1, 2, 2),
        # Community B: nodes 3, 4, 5 (densely connected at times 0-2)
        (3, 4, 0), (4, 5, 0), (3, 5, 0),
        (3, 4, 1), (4, 5, 1), (3, 5, 1),
        (3, 4, 2), (4, 5, 2),
        # Bridge between communities at time 2
        (2, 3, 2),
    ])
    return ls

ls = create_sample_linkstream()
print(f"Created linkstream with {ls.nb_nodes} nodes and {ls.nb_time_edges} time-edges")

## 1. Basic Community Detection

Detect communities with default parameters.

In [ ]:
# Detect communities with default parameters
communities = lago_modules(ls)

print(f"Found {communities.nb_modules} communities")
print(f"Over {communities.nb_times} time steps")
print(f"Involving {communities.nb_nodes} nodes")

### List all communities

In [ ]:
print("Community details:")
for module in communities.iter_modules():
    print(f"  Community {module.label}:")
    print(f"    Nodes: {module.nodes}")
    print(f"    Duration: {module.duration} time steps")
    print(f"    Cohesion: {module.cohesion:.2f}")

## 2. Node Trajectory

Track how a node's community membership changes over time.

In [ ]:
# Track node 0
node_id = 0
print(f"Tracking node {node_id}:")

# Get all memberships
memberships = communities.get_modules_of_node(node_id)
for m in memberships:
    print(f"  In community {m.module} during: {sorted(m.times)}")

In [ ]:
# Get trajectory over time
trajectory = communities.get_node_trajectory(node_id)
print(f"Time -> Community: {trajectory}")

# Stability score (1.0 = never switches, lower = more switches)
stability = communities.get_stability_score(node_id)
print(f"Stability score: {stability:.2f}")

## 3. Time Snapshots

Get community structure at specific time points.

In [ ]:
for time in range(3):
    print(f"\nAt time {time}:")
    
    # Which nodes are in which community?
    membership = communities.get_nodes_modules_membership_at_time(time)
    for node, comm in sorted(membership.items()):
        print(f"  Node {node} -> Community {comm}")

## 4. Parameter Comparison

Compare different parameter settings to understand their effects.

### Effect of omega (temporal smoothness)

Higher omega values encourage more stable communities over time.

In [ ]:
print("Effect of omega (temporal smoothness):")
for omega in [0.5, 2, 5]:
    comms = lago_modules(ls, omega=omega)
    print(f"  omega={omega}: {comms.nb_modules} communities")

### Effect of gamma (resolution)

Higher gamma values favor smaller, more focused communities.

In [ ]:
print("Effect of gamma (resolution):")
for gamma in [0.5, 1, 2]:
    comms = lago_modules(ls, gamma=gamma)
    print(f"  gamma={gamma}: {comms.nb_modules} communities")

### Effect of lex (expectation type)

- **MM (Mean-Membership):** Most flexible, general use (default)
- **JM (Joint-Membership):** Favors stable, long-lasting communities

In [ ]:
print("Effect of lex (expectation type):")
for lex in ["MM", "JM"]:
    comms = lago_modules(ls, lex=lex)
    print(f"  lex='{lex}': {comms.nb_modules} communities")

## 5. Save and Load Communities

You can save communities to files and load them later.

In [ ]:
# Example usage (uncomment to run):
print("Saving communities to files:")
print('  communities.to_json("communities.json")')
print('  communities.to_csv("communities.csv")')
print('  communities.to_txt("communities.txt")')
print()
print("Loading communities from files:")
print('  from lago import TimeModules')
print('  tm = TimeModules(path="communities.json")')

## 6. Module Analysis

Analyze individual modules in detail.

In [ ]:
# Get a specific module
if communities.nb_modules > 0:
    module = communities.get_module(0)
    
    print(f"Analyzing Module {module.label}:")
    print(f"  Nodes: {module.nodes}")
    print(f"  Times: {module.times}")
    print(f"  Size: {module.size} nodes")
    print(f"  Duration: {module.duration} time steps")
    print(f"  Time range: {module.time_range}")
    print(f"  Cohesion: {module.cohesion:.2f}")

In [ ]:
# Node segments within module
if communities.nb_modules > 0:
    module = communities.get_module(0)
    segments = module.get_node_segments()
    print("Node time segments:")
    for node, segs in segments.items():
        print(f"  Node {node}: {[(s.start, s.end) for s in segs]}")

---

## Next Steps

Continue with:
- [04_modularity.ipynb](04_modularity.ipynb) - Understand quality metrics and modularity
- [05_visualization.ipynb](05_visualization.ipynb) - Create visualizations of your communities

**Practical guides:**
- [real_world_preprocessing.ipynb](real_world_preprocessing.ipynb) - Work with real-world data (names, dates)